# Descarga auditable Sentinel-2 L2A — Salar de Olaroz

Este notebook genera productos **rectangulares** para una práctica de análisis de datos y aprendizaje automático clásico:

1. **Raster espectral multibanda rectangular** con mosaico de época seca, en orden RGB natural:
   - B04, B03, B02, B05, B06, B07, B08, B8A, B11, B12
2. **Raster SCL rectangular aparte** para control de calidad:
   - SCL

El raster espectral se genera como **mosaico temporal por mediana** usando varios items Sentinel-2 L2A de la ventana seca para reducir huecos y evitar NoData cuando el AOI cae sobre más de un tile.

Cambio importante: para que el resultado quede perfectamente rectangular en QGIS, **no se usa `rio.clip()` con geometrías poligonales**. El recorte rectangular se controla con `bounds_latlon` en `stackstac.stack()`.


In [1]:
import os
from pathlib import Path
import json
from collections import defaultdict

import pandas as pd
import geopandas as gpd
import numpy as np

from shapely.geometry import Polygon, box, mapping, shape
from shapely.ops import unary_union

from pystac_client import Client
import planetary_computer as pc
import stackstac
import rioxarray
import rasterio


In [2]:
# ======================================================
# 1. Configuración general
# ======================================================

# Si estás ejecutando desde /scripts, BASE_DIR será la carpeta padre.
# Si estás ejecutando desde la raíz del proyecto, podés cambiarlo por:
# BASE_DIR = Path.cwd()
BASE_DIR = Path(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

RASTER_DIR = BASE_DIR / "raster"
LABELS_DIR = BASE_DIR / "labels"
METADATA_DIR = BASE_DIR / "metadata"

for folder in [RASTER_DIR, LABELS_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RASTER_DIR:", RASTER_DIR)
print("LABELS_DIR:", LABELS_DIR)
print("METADATA_DIR:", METADATA_DIR)

BASE_DIR: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis
RASTER_DIR: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\raster
LABELS_DIR: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\labels
METADATA_DIR: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata


In [3]:
# ======================================================
# 2. AOI preliminar - Salar de Olaroz
# ======================================================
# AOI amplio para incluir salar, bordes, zonas húmedas,
# infraestructura, piletas y suelos periféricos.
#
# Importante para este notebook:
# - Se guarda un AOI de referencia como GeoJSON.
# - El raster final se exporta rectangular.
# - Para lograrlo, NO se recorta con rio.clip() al final.
# - Se usa el bounding box rectangular del AOI mediante bounds_latlon.

olaroz_polygon = Polygon([
    (-66.86, -23.68),
    (-66.50, -23.68),
    (-66.50, -23.25),
    (-66.86, -23.25),
    (-66.86, -23.68),
])

aoi_gdf = gpd.GeoDataFrame(
    {
        "name": ["Salar de Olaroz - AOI preliminar"],
        "purpose": ["Mentoria - clasificación clásica de cobertura terrestre"],
    },
    geometry=[olaroz_polygon],
    crs="EPSG:4326",
)

aoi_path = LABELS_DIR / "olaroz_aoi.geojson"
aoi_gdf.to_file(aoi_path, driver="GeoJSON")

# AOI rectangular explícito a partir del bounding box del AOI.
# Este GeoJSON sirve para documentar el rectángulo exacto usado como ventana de raster.
minx, miny, maxx, maxy = aoi_gdf.total_bounds
olaroz_rect_polygon = box(minx, miny, maxx, maxy)

aoi_rect_wgs84 = gpd.GeoDataFrame(
    {
        "name": ["Salar de Olaroz - AOI rectangular"],
        "purpose": ["Ventana rectangular para raster Sentinel-2 de la diplomatura"],
    },
    geometry=[olaroz_rect_polygon],
    crs="EPSG:4326",
)

aoi_rect_path = LABELS_DIR / "olaroz_aoi_rectangular.geojson"
aoi_rect_wgs84.to_file(aoi_rect_path, driver="GeoJSON")

print(f"AOI guardado en: {aoi_path}")
print(f"AOI rectangular guardado en: {aoi_rect_path}")
aoi_rect_wgs84


AOI guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\labels\olaroz_aoi.geojson
AOI rectangular guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\labels\olaroz_aoi_rectangular.geojson


,name,purpose,geometry
0,Salar de Olaroz - AOI rectangular,Ventana rectangular para raster Sentinel-2 de ...,"POLYGON ((-66.5 -23.68, -66.5 -23.25, -66.86 -..."


In [4]:
# ======================================================
# 3. Parámetros auditables de búsqueda
# ======================================================

COLLECTION = "sentinel-2-l2a"
DATE_RANGE = "2025-06-01/2025-09-30"   # Época seca: junio a septiembre
MAX_CLOUD = 20                         # Nubosidad máxima permitida (%)

TARGET_EPSG = 32719                    # UTM 19S para Olaroz
RESOLUTION = 20                        # 20 m: grilla común para bandas 10 m y 20 m

# Bandas espectrales en orden RGB natural al abrir en QGIS:
# Banda 1 = B04 = Rojo
# Banda 2 = B03 = Verde
# Banda 3 = B02 = Azul
SPECTRAL_BANDS = [
    "B04",   # Red / Rojo - 10 m
    "B03",   # Green / Verde - 10 m
    "B02",   # Blue / Azul - 10 m
    "B05",   # Vegetation Red Edge 1 - 20 m
    "B06",   # Vegetation Red Edge 2 - 20 m
    "B07",   # Vegetation Red Edge 3 - 20 m
    "B08",   # NIR - 10 m
    "B8A",   # Narrow NIR - 20 m
    "B11",   # SWIR 1 - 20 m
    "B12",   # SWIR 2 - 20 m
]

# SCL se exporta aparte porque es una capa categórica, no una banda continua.
# No conviene calcular mediana de SCL junto con las bandas espectrales.
SCL_BANDS = ["SCL"]

# Para evitar cargas excesivas, se seleccionan varias fechas buenas, no necesariamente todas.
# El mosaico se arma con varios items de la ventana seca.
MIN_DATES_FOR_MOSAIC = 3
MAX_DATES_FOR_MOSAIC = 6
COVERAGE_TARGET = 99.0

print("Bandas espectrales:", SPECTRAL_BANDS)
print("SCL aparte:", SCL_BANDS)


Bandas espectrales: ['B04', 'B03', 'B02', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
SCL aparte: ['SCL']


In [5]:
# ======================================================
# 4. Consulta STAC
# ======================================================

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

search = catalog.search(
    collections=[COLLECTION],
    intersects=mapping(olaroz_polygon),
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    max_items=300,
)

items = list(search.items())

if not items:
    raise RuntimeError(
        "No se encontraron imágenes con esos filtros. "
        "Probá ampliar DATE_RANGE o subir MAX_CLOUD."
    )

print(f"Items candidatos encontrados: {len(items)}")


Items candidatos encontrados: 56


In [6]:
# ======================================================
# 5. Guardar metadatos de todas las escenas candidatas
# ======================================================

records = []

for item in items:
    props = item.properties

    records.append({
        "item_id": item.id,
        "datetime": props.get("datetime"),
        "date": props.get("datetime", "")[:10],
        "platform": props.get("platform"),
        "constellation": props.get("constellation"),
        "mgrs_tile": props.get("s2:mgrs_tile"),
        "eo_cloud_cover": props.get("eo:cloud_cover"),
        "processing_baseline": props.get("s2:processing_baseline"),
        "collection": COLLECTION,
        "date_range_filter": DATE_RANGE,
        "cloud_filter": MAX_CLOUD,
    })

metadata_df = pd.DataFrame(records)
metadata_df = metadata_df.sort_values(["date", "eo_cloud_cover"], ascending=[True, True])

candidates_csv = METADATA_DIR / "sentinel2_olaroz_candidate_scenes.csv"
metadata_df.to_csv(candidates_csv, index=False, encoding="utf-8")

print(f"Metadatos de escenas candidatas guardados en: {candidates_csv}")
metadata_df.head(20)


Metadatos de escenas candidatas guardados en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\sentinel2_olaroz_candidate_scenes.csv


,item_id,datetime,date,platform,constellation,mgrs_tile,eo_cloud_cover,processing_baseline,collection,date_range_filter,cloud_filter
55,S2C_MSIL2A_20250603T142731_R053_T19KGQ_2025060...,2025-06-03T14:27:31.025000Z,2025-06-03,Sentinel-2C,Sentinel 2,19KGQ,17.226343,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
53,S2A_MSIL2A_20250605T142811_R053_T19KGQ_2025060...,2025-06-05T14:28:11.024000Z,2025-06-05,Sentinel-2A,Sentinel 2,19KGQ,0.153477,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
54,S2A_MSIL2A_20250605T142811_R053_T19KGP_2025060...,2025-06-05T14:28:11.024000Z,2025-06-05,Sentinel-2A,Sentinel 2,19KGP,0.242289,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
51,S2B_MSIL2A_20250608T142709_R053_T19KGQ_2025060...,2025-06-08T14:27:09.024000Z,2025-06-08,Sentinel-2B,Sentinel 2,19KGQ,0.061957,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
52,S2B_MSIL2A_20250608T142709_R053_T19KGP_2025060...,2025-06-08T14:27:09.024000Z,2025-06-08,Sentinel-2B,Sentinel 2,19KGP,0.124018,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
49,S2C_MSIL2A_20250613T142731_R053_T19KGQ_2025061...,2025-06-13T14:27:31.025000Z,2025-06-13,Sentinel-2C,Sentinel 2,19KGQ,0.091128,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
50,S2C_MSIL2A_20250613T142731_R053_T19KGP_2025061...,2025-06-13T14:27:31.025000Z,2025-06-13,Sentinel-2C,Sentinel 2,19KGP,0.457085,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
47,S2B_MSIL2A_20250618T142709_R053_T19KGQ_2025061...,2025-06-18T14:27:09.024000Z,2025-06-18,Sentinel-2B,Sentinel 2,19KGQ,0.089349,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
48,S2B_MSIL2A_20250618T142709_R053_T19KGP_2025061...,2025-06-18T14:27:09.024000Z,2025-06-18,Sentinel-2B,Sentinel 2,19KGP,5.602549,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
45,S2C_MSIL2A_20250623T142731_R053_T19KGQ_2025062...,2025-06-23T14:27:31.025000Z,2025-06-23,Sentinel-2C,Sentinel 2,19KGQ,0.164774,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20


In [7]:
# ======================================================
# 6. Analizar cobertura por fecha
# ======================================================
# Esta etapa evita usar una sola escena. Agrupa los items por fecha
# y calcula qué porcentaje del AOI rectangular cubre la unión de tiles/items.

items_by_date = defaultdict(list)

for item in items:
    date = item.properties["datetime"][:10]
    items_by_date[date].append(item)

# A partir de aquí, aoi_wgs84 representa la ventana rectangular de trabajo.
aoi_wgs84 = aoi_rect_wgs84.to_crs("EPSG:4326")
aoi_utm = aoi_wgs84.to_crs(f"EPSG:{TARGET_EPSG}")
aoi_area = aoi_utm.geometry.iloc[0].area

coverage_records = []

for date, date_items in items_by_date.items():
    geoms = [shape(item.geometry) for item in date_items]

    tiles_gdf = gpd.GeoDataFrame(
        {
            "item_id": [item.id for item in date_items],
            "eo_cloud_cover": [
                item.properties.get("eo:cloud_cover", np.nan)
                for item in date_items
            ],
        },
        geometry=geoms,
        crs="EPSG:4326",
    ).to_crs(f"EPSG:{TARGET_EPSG}")

    union_geom = unary_union(tiles_gdf.geometry)
    covered_area = union_geom.intersection(aoi_utm.geometry.iloc[0]).area
    coverage_pct = covered_area / aoi_area * 100

    cloud_values = [
        item.properties.get("eo:cloud_cover", np.nan)
        for item in date_items
    ]

    coverage_records.append({
        "date": date,
        "n_items": len(date_items),
        "coverage_pct": coverage_pct,
        "mean_cloud": float(np.nanmean(cloud_values)),
        "min_cloud": float(np.nanmin(cloud_values)),
        "max_cloud": float(np.nanmax(cloud_values)),
        "item_ids": ";".join([item.id for item in date_items]),
        "mgrs_tiles": ";".join(sorted(set([
            str(item.properties.get("s2:mgrs_tile"))
            for item in date_items
        ]))),
    })

coverage_df = pd.DataFrame(coverage_records)

coverage_df = coverage_df.sort_values(
    ["coverage_pct", "mean_cloud"],
    ascending=[False, True]
)

coverage_csv = METADATA_DIR / "sentinel2_olaroz_coverage_by_date.csv"
coverage_df.to_csv(coverage_csv, index=False, encoding="utf-8")

print(f"Cobertura por fecha guardada en: {coverage_csv}")
coverage_df.head(20)


Cobertura por fecha guardada en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\sentinel2_olaroz_coverage_by_date.csv


,date,n_items,coverage_pct,mean_cloud,min_cloud,max_cloud,item_ids,mgrs_tiles
26,2025-06-08,2,100.0,0.092988,0.061957,0.124018,S2B_MSIL2A_20250608T142709_R053_T19KGQ_2025060...,19KGP;19KGQ
1,2025-09-21,2,100.0,0.143778,0.035975,0.251582,S2C_MSIL2A_20250921T142731_R053_T19KGQ_2025092...,19KGP;19KGQ
15,2025-07-28,2,100.0,0.169023,0.084976,0.253071,S2B_MSIL2A_20250728T142719_R053_T19KGQ_2025072...,19KGP;19KGQ
13,2025-08-04,2,100.0,0.174073,0.066344,0.281801,S2A_MSIL2A_20250804T142811_R053_T19KGQ_2025080...,19KGP;19KGQ
27,2025-06-05,2,100.0,0.197883,0.153477,0.242289,S2A_MSIL2A_20250605T142811_R053_T19KGQ_2025060...,19KGP;19KGQ
17,2025-07-18,2,100.0,0.238682,0.102223,0.375141,S2B_MSIL2A_20250718T142719_R053_T19KGQ_2025071...,19KGP;19KGQ
25,2025-06-13,2,100.0,0.274107,0.091128,0.457085,S2C_MSIL2A_20250613T142731_R053_T19KGQ_2025061...,19KGP;19KGQ
21,2025-07-03,2,100.0,0.294180,0.128371,0.459989,S2C_MSIL2A_20250703T142811_R053_T19KGQ_2025070...,19KGP;19KGQ
23,2025-06-23,2,100.0,0.295621,0.164774,0.426468,S2C_MSIL2A_20250623T142731_R053_T19KGQ_2025062...,19KGP;19KGQ
11,2025-08-12,2,100.0,0.295891,0.258824,0.332959,S2C_MSIL2A_20250812T142731_R053_T19KGQ_2025081...,19KGP;19KGQ


In [8]:
# ======================================================
# 7. Seleccionar varias fechas/items para el mosaico
# ======================================================
# Criterio:
# 1) Prioriza fechas con alta cobertura del AOI.
# 2) Entre fechas similares, prioriza menor nubosidad.
# 3) Usa varias fechas de la ventana seca para reducir NoData/huecos.
#
# Si hay fechas con cobertura >= COVERAGE_TARGET, toma las mejores.
# Si no, toma las mejores fechas disponibles.

good_dates_df = coverage_df[coverage_df["coverage_pct"] >= COVERAGE_TARGET].copy()

if len(good_dates_df) > 0:
    selected_dates_df = good_dates_df.sort_values(
        ["mean_cloud", "coverage_pct"],
        ascending=[True, False]
    ).head(MAX_DATES_FOR_MOSAIC)
else:
    selected_dates_df = coverage_df.sort_values(
        ["coverage_pct", "mean_cloud"],
        ascending=[False, True]
    ).head(MAX_DATES_FOR_MOSAIC)

# Garantizar un mínimo de fechas si existen
if len(selected_dates_df) < MIN_DATES_FOR_MOSAIC and len(coverage_df) >= MIN_DATES_FOR_MOSAIC:
    selected_dates_df = coverage_df.sort_values(
        ["coverage_pct", "mean_cloud"],
        ascending=[False, True]
    ).head(MIN_DATES_FOR_MOSAIC)

selected_dates = selected_dates_df["date"].tolist()

selected_items = []
for date in selected_dates:
    selected_items.extend(items_by_date[date])

selected_items_df = metadata_df[metadata_df["item_id"].isin([item.id for item in selected_items])].copy()

selected_items_csv = METADATA_DIR / "sentinel2_olaroz_selected_items_for_mosaic.csv"
selected_items_df.to_csv(selected_items_csv, index=False, encoding="utf-8")

selected_items_json = METADATA_DIR / "sentinel2_olaroz_selected_items_for_mosaic.json"
with open(selected_items_json, "w", encoding="utf-8") as f:
    json.dump(
        [item.to_dict() for item in selected_items],
        f,
        ensure_ascii=False,
        indent=2
    )

print("Fechas seleccionadas para el mosaico:", selected_dates)
print("Cantidad de items seleccionados:", len(selected_items))
print(f"CSV de items seleccionados: {selected_items_csv}")
print(f"JSON STAC de items seleccionados: {selected_items_json}")

selected_items_df


Fechas seleccionadas para el mosaico: ['2025-06-08', '2025-09-21', '2025-07-28', '2025-08-04', '2025-06-05', '2025-07-18']
Cantidad de items seleccionados: 12
CSV de items seleccionados: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\sentinel2_olaroz_selected_items_for_mosaic.csv
JSON STAC de items seleccionados: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\sentinel2_olaroz_selected_items_for_mosaic.json


,item_id,datetime,date,platform,constellation,mgrs_tile,eo_cloud_cover,processing_baseline,collection,date_range_filter,cloud_filter
53,S2A_MSIL2A_20250605T142811_R053_T19KGQ_2025060...,2025-06-05T14:28:11.024000Z,2025-06-05,Sentinel-2A,Sentinel 2,19KGQ,0.153477,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
54,S2A_MSIL2A_20250605T142811_R053_T19KGP_2025060...,2025-06-05T14:28:11.024000Z,2025-06-05,Sentinel-2A,Sentinel 2,19KGP,0.242289,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
51,S2B_MSIL2A_20250608T142709_R053_T19KGQ_2025060...,2025-06-08T14:27:09.024000Z,2025-06-08,Sentinel-2B,Sentinel 2,19KGQ,0.061957,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
52,S2B_MSIL2A_20250608T142709_R053_T19KGP_2025060...,2025-06-08T14:27:09.024000Z,2025-06-08,Sentinel-2B,Sentinel 2,19KGP,0.124018,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
32,S2B_MSIL2A_20250718T142719_R053_T19KGQ_2025071...,2025-07-18T14:27:19.024000Z,2025-07-18,Sentinel-2B,Sentinel 2,19KGQ,0.102223,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
33,S2B_MSIL2A_20250718T142719_R053_T19KGP_2025071...,2025-07-18T14:27:19.024000Z,2025-07-18,Sentinel-2B,Sentinel 2,19KGP,0.375141,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
28,S2B_MSIL2A_20250728T142719_R053_T19KGQ_2025072...,2025-07-28T14:27:19.024000Z,2025-07-28,Sentinel-2B,Sentinel 2,19KGQ,0.084976,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
29,S2B_MSIL2A_20250728T142719_R053_T19KGP_2025072...,2025-07-28T14:27:19.024000Z,2025-07-28,Sentinel-2B,Sentinel 2,19KGP,0.253071,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
24,S2A_MSIL2A_20250804T142811_R053_T19KGQ_2025080...,2025-08-04T14:28:11.024000Z,2025-08-04,Sentinel-2A,Sentinel 2,19KGQ,0.066344,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20
25,S2A_MSIL2A_20250804T142811_R053_T19KGP_2025080...,2025-08-04T14:28:11.024000Z,2025-08-04,Sentinel-2A,Sentinel 2,19KGP,0.281801,05.11,sentinel-2-l2a,2025-06-01/2025-09-30,20


In [9]:
# ======================================================
# 8. Validar que las bandas existan en todos los items seleccionados
# ======================================================

required_assets = SPECTRAL_BANDS + SCL_BANDS

missing_by_item = {}

for item in selected_items:
    missing = [asset for asset in required_assets if asset not in item.assets]
    if missing:
        missing_by_item[item.id] = missing

if missing_by_item:
    print("Hay assets faltantes:")
    for item_id, missing in missing_by_item.items():
        print(item_id, missing)
    raise ValueError("Algunos items no tienen todas las bandas requeridas.")

print("Todos los items seleccionados contienen las bandas requeridas.")


Todos los items seleccionados contienen las bandas requeridas.


In [10]:
# ======================================================
# 9. Crear mosaico espectral rectangular por mediana temporal
# ======================================================
# Este es el raster principal para la diplomatura.
# Primeras 3 bandas: B04, B03, B02 => RGB natural en QGIS.
#
# Se usa dtype float32 y fill_value=np.float32(np.nan) para distinguir NoData real.
# La mediana temporal usa varios items de la ventana seca.
#
# Cambio clave para salida rectangular:
# - Se usa bounds_latlon=bounds en stackstac.
# - NO se aplica rio.clip() con geometrías poligonales.

signed_items = [pc.sign(item) for item in selected_items]

# Bounds rectangulares en EPSG:4326.
# Esto define la ventana rectangular del raster final.
bounds = aoi_wgs84.total_bounds  # minx, miny, maxx, maxy

spectral_stack = stackstac.stack(
    signed_items,
    assets=SPECTRAL_BANDS,
    bounds_latlon=bounds,
    epsg=TARGET_EPSG,
    resolution=RESOLUTION,
    dtype="float32",
    fill_value=np.float32(np.nan),
    rescale=False,
)

# Mosaico temporal por mediana.
# dims esperadas: time, band, y, x
spectral_mosaic = spectral_stack.median(dim="time", skipna=True)

# Restaurar dimensiones espaciales y CRS para rioxarray
spectral_mosaic = spectral_mosaic.rio.set_spatial_dims(
    x_dim="x",
    y_dim="y",
    inplace=False,
)

spectral_mosaic = spectral_mosaic.rio.write_crs(
    f"EPSG:{TARGET_EPSG}",
    inplace=False,
)

# IMPORTANTE:
# No usamos rio.clip(). El mosaico ya está limitado por bounds_latlon
# y por lo tanto queda rectangular en la grilla de salida.
spectral_clip = spectral_mosaic.compute()
spectral_clip = spectral_clip.transpose("band", "y", "x")

# Definir NoData
spectral_clip = spectral_clip.rio.write_nodata(
    np.float32(np.nan),
    inplace=False,
)

print("Mosaico espectral rectangular generado.")
print("Dimensiones:", spectral_clip.dims)
print("Tamaños:", spectral_clip.sizes)
print("Bandas:", [str(b) for b in spectral_clip.coords["band"].values])


Mosaico espectral rectangular generado.
Dimensiones: ('band', 'y', 'x')
Tamaños: Frozen({'band': 10, 'y': 2412, 'x': 1879})
Bandas: ['B04', 'B03', 'B02', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']


In [11]:
# ======================================================
# 10. Exportar raster espectral rectangular
# ======================================================

spectral_out_tif = RASTER_DIR / "olaroz_sentinel2_l2a_seca_2025_mosaico_rgb_natural_rectangular.tif"

spectral_clip.rio.to_raster(spectral_out_tif)

spectral_band_names = [str(b) for b in spectral_clip.coords["band"].values]

with rasterio.open(spectral_out_tif, "r+") as dst:
    for idx, band_name in enumerate(spectral_band_names, start=1):
        dst.set_band_description(idx, band_name)

    dst.update_tags(
        area="Salar de Olaroz, Jujuy, Argentina",
        collection=COLLECTION,
        date_range_filter=DATE_RANGE,
        cloud_filter=str(MAX_CLOUD),
        selected_dates=",".join(selected_dates),
        selected_items=",".join([item.id for item in selected_items]),
        purpose="Diplomatura - clasificación clásica de cobertura terrestre",
        bands=",".join(spectral_band_names),
        resolution_meters=str(RESOLUTION),
        method="Mosaico temporal rectangular por mediana usando varios items Sentinel-2 L2A de época seca",
        qgis_rgb="Banda 1=B04, Banda 2=B03, Banda 3=B02",
        rectangular_output="true",
        clipping="No se aplicó rio.clip(); la ventana rectangular se definió con bounds_latlon",
        nodata_value="NaN",
    )

print(f"Raster espectral rectangular guardado en: {spectral_out_tif}")
print("Abrir en QGIS como RGB natural: Banda 1, Banda 2, Banda 3.")


Raster espectral rectangular guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\raster\olaroz_sentinel2_l2a_seca_2025_mosaico_rgb_natural_rectangular.tif
Abrir en QGIS como RGB natural: Banda 1, Banda 2, Banda 3.


In [12]:
# ======================================================
# 11. Crear SCL rectangular aparte para control de calidad
# ======================================================
# SCL es una capa categórica. Por eso NO se calcula mediana.
#
# Estrategia:
# - Se apila SCL de los mismos items usados en el mosaico espectral.
# - Se toma el primer valor SCL válido distinto de 0 para cada píxel.
# - Si ningún item tiene valor válido, queda 0 = NoData.
# - No se aplica rio.clip(), para que el resultado quede rectangular.
#
# Valores SCL:
# 0  = NoData
# 1  = Saturated/defective
# 2  = Dark area pixels
# 3  = Cloud shadows
# 4  = Vegetation
# 5  = Not vegetated / bare soil
# 6  = Water
# 7  = Unclassified
# 8  = Cloud medium probability
# 9  = Cloud high probability
# 10 = Thin cirrus
# 11 = Snow / ice

SCL_BANDS = ["SCL"]

scl_stack = stackstac.stack(
    signed_items,
    assets=SCL_BANDS,
    bounds_latlon=bounds,
    epsg=TARGET_EPSG,
    resolution=RESOLUTION,
    dtype="uint8",
    fill_value=np.uint8(0),
    rescale=False,
)

# Seleccionar la única banda SCL
# dims esperadas: time, y, x
scl = scl_stack.sel(band="SCL")

# Restaurar dimensiones espaciales y CRS para rioxarray
scl = scl.rio.set_spatial_dims(
    x_dim="x",
    y_dim="y",
    inplace=False,
)

scl = scl.rio.write_crs(
    f"EPSG:{TARGET_EPSG}",
    inplace=False,
)

# Máscara de valores válidos
# 0 = NoData, por eso buscamos valores distintos de 0
valid = scl != 0

# Primer índice temporal donde SCL es válido
first_valid_idx = valid.argmax(dim="time")

# Saber qué píxeles tuvieron al menos un valor válido en algún item
has_valid = valid.any(dim="time")

# ------------------------------------------------------
# Corrección importante:
# xarray no permite isel() con índices Dask multidimensionales.
# Por eso computamos first_valid_idx y has_valid antes de isel().
# ------------------------------------------------------
first_valid_idx = first_valid_idx.compute()
has_valid = has_valid.compute()

# También computamos SCL porque es una sola banda uint8,
# mucho más liviana que el mosaico espectral
scl = scl.compute()

# Tomar el primer valor SCL válido por píxel
scl_control = scl.isel(time=first_valid_idx)

# Donde nunca hubo valor válido, dejar 0 = NoData
scl_control = scl_control.where(has_valid, 0).astype("uint8")

# Restaurar CRS nuevamente después de isel/where
scl_control = scl_control.rio.set_spatial_dims(
    x_dim="x",
    y_dim="y",
    inplace=False,
)

scl_control = scl_control.rio.write_crs(
    f"EPSG:{TARGET_EPSG}",
    inplace=False,
)

# IMPORTANTE:
# No usamos rio.clip(). El SCL queda con la misma grilla rectangular
# del raster espectral porque usa los mismos bounds_latlon, EPSG y resolución.
scl_clip = scl_control.compute()

# Asegurar que quede como raster de 1 banda: band, y, x
if "band" not in scl_clip.dims:
    scl_clip = scl_clip.expand_dims(band=["SCL"])

scl_clip = scl_clip.transpose("band", "y", "x")

# Definir NoData
scl_clip = scl_clip.rio.write_nodata(
    np.uint8(0),
    inplace=False,
)

print("SCL de control rectangular generado.")
print("Dimensiones:", scl_clip.dims)
print("Tamaños:", scl_clip.sizes)


SCL de control rectangular generado.
Dimensiones: ('band', 'y', 'x')
Tamaños: Frozen({'band': 1, 'y': 2412, 'x': 1879})


In [13]:
# ======================================================
# 12. Exportar SCL rectangular aparte
# ======================================================

scl_out_tif = RASTER_DIR / "olaroz_sentinel2_l2a_seca_2025_scl_control_rectangular.tif"

scl_clip.rio.to_raster(scl_out_tif)

with rasterio.open(scl_out_tif, "r+") as dst:
    dst.set_band_description(1, "SCL")
    dst.nodata = 0

    dst.update_tags(
        area="Salar de Olaroz, Jujuy, Argentina",
        collection=COLLECTION,
        date_range_filter=DATE_RANGE,
        cloud_filter=str(MAX_CLOUD),
        selected_dates=",".join(selected_dates),
        selected_items=",".join([item.id for item in selected_items]),
        purpose="Control de calidad para evitar nubes, sombras y NoData",
        band="SCL",
        resolution_meters=str(RESOLUTION),
        method="Primer valor SCL válido distinto de 0 usando los mismos items del mosaico espectral",
        rectangular_output="true",
        clipping="No se aplicó rio.clip(); la ventana rectangular se definió con bounds_latlon",
        scl_0="NoData",
        scl_1="Saturated or defective",
        scl_2="Dark area pixels",
        scl_3="Cloud shadows",
        scl_4="Vegetation",
        scl_5="Bare soil / not vegetated",
        scl_6="Water",
        scl_7="Unclassified",
        scl_8="Cloud medium probability",
        scl_9="Cloud high probability",
        scl_10="Thin cirrus",
        scl_11="Snow or ice",
    )

print(f"Raster SCL rectangular guardado en: {scl_out_tif}")


Raster SCL rectangular guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\raster\olaroz_sentinel2_l2a_seca_2025_scl_control_rectangular.tif


In [14]:
# ======================================================
# 13. Control rápido de NoData, grilla y valores válidos
# ======================================================

with rasterio.open(spectral_out_tif) as src:
    print("Raster espectral:")
    print("  Archivo:", spectral_out_tif)
    print("  CRS:", src.crs)
    print("  Resolución:", src.res)
    print("  Tamaño columnas x filas:", src.width, "x", src.height)
    print("  Bandas:", src.count)
    print("  Descripciones:", src.descriptions)
    print("  NoData:", src.nodata)
    first_band = src.read(1, masked=True)
    print("  Pixeles válidos banda 1:", int(first_band.count()))
    print("  Pixeles totales:", first_band.size)

with rasterio.open(scl_out_tif) as src:
    print("
Raster SCL:")
    print("  Archivo:", scl_out_tif)
    print("  CRS:", src.crs)
    print("  Resolución:", src.res)
    print("  Tamaño columnas x filas:", src.width, "x", src.height)
    print("  Bandas:", src.count)
    print("  Descripciones:", src.descriptions)
    print("  NoData:", src.nodata)
    scl_arr = src.read(1, masked=False)
    unique_vals = sorted(np.unique(scl_arr).tolist())
    print("  Valores SCL presentes:", unique_vals)

print("
Nota: ambos rasters deben tener el mismo tamaño, CRS y resolución.")


SyntaxError: EOL while scanning string literal (2232851569.py, line 19)

In [ ]:
# ======================================================
# 14. Crear protocolo de adquisición en Markdown
# ======================================================

protocol = f"""# Protocolo de adquisición - Salar de Olaroz

## Área de estudio
Salar de Olaroz, departamento de Susques, provincia de Jujuy, Argentina.

## Fuente de datos
Microsoft Planetary Computer STAC API.

## Colección
`{COLLECTION}`

## Producto
Sentinel-2 Level-2A Surface Reflectance.

## Criterios de búsqueda
- AOI de referencia: `labels/{aoi_path.name}`
- AOI rectangular usado como ventana de raster: `labels/{aoi_rect_path.name}`
- Rango temporal: `{DATE_RANGE}`
- Época: seca
- Nubosidad máxima por item: `{MAX_CLOUD}%`
- CRS de trabajo: `EPSG:{TARGET_EPSG}`
- Resolución de trabajo: `{RESOLUTION}` metros

## Estrategia para evitar NoData
No se utilizó una única escena Sentinel-2. Se consultaron múltiples items dentro de la ventana seca y se seleccionaron varias fechas/items con buena cobertura espacial y baja nubosidad.  
El raster espectral final se construyó mediante un mosaico temporal por mediana, usando los items seleccionados.

## Estrategia para salida rectangular
El producto final se exportó como raster rectangular.  
Para lograrlo, no se aplicó `rio.clip()` con geometrías poligonales al final. La ventana rectangular se definió mediante `bounds_latlon` en `stackstac.stack()`, usando el bounding box del AOI.

## Raster espectral principal
Archivo:
`raster/{spectral_out_tif.name}`

Bandas, en orden:
`{",".join(SPECTRAL_BANDS)}`

El raster está ordenado para abrir en QGIS como RGB natural:
- Banda 1 = B04 = Rojo
- Banda 2 = B03 = Verde
- Banda 3 = B02 = Azul

## Raster SCL de control de calidad
Archivo:
`raster/{scl_out_tif.name}`

La capa SCL se exportó por separado porque es categórica y no debe mezclarse con las bandas espectrales mediante mediana.  
Debe utilizarse para evitar muestras sobre nubes, sombras, píxeles no clasificados o NoData.

Clases SCL principales:
- 0: NoData
- 1: Saturated/defective
- 2: Dark area pixels
- 3: Cloud shadows
- 4: Vegetation
- 5: Bare soil / not vegetated
- 6: Water
- 7: Unclassified
- 8: Cloud medium probability
- 9: Cloud high probability
- 10: Thin cirrus
- 11: Snow/ice

## Archivos de metadatos
- Escenas candidatas: `metadata/{candidates_csv.name}`
- Cobertura por fecha: `metadata/{coverage_csv.name}`
- Items usados en el mosaico: `metadata/{selected_items_csv.name}`
- STAC JSON de items usados: `metadata/{selected_items_json.name}`

## Uso previsto
Este dataset se utilizará para una práctica de análisis de datos y aprendizaje automático clásico.  
Los alumnos deberán construir un dataset tabular a partir de polígonos GeoJSON etiquetados, bandas espectrales e índices derivados.

## Recomendación para muestreo
Evitar polígonos sobre:
- SCL = 0 NoData
- SCL = 1 Saturated/defective
- SCL = 3 Cloud shadows
- SCL = 7 Unclassified
- SCL = 8 Cloud medium probability
- SCL = 9 Cloud high probability
- SCL = 10 Thin cirrus
- SCL = 11 Snow/ice

Para esta práctica, priorizar polígonos homogéneos y confiables sobre clases de cobertura terrestre claramente visibles.
"""

protocol_path = METADATA_DIR / "acquisition_protocol.md"
protocol_path.write_text(protocol, encoding="utf-8")

print(f"Protocolo guardado en: {protocol_path}")
print("Proceso finalizado correctamente.")
